# loop-autopsy demo

Run the cells from top to bottom. Each broken loop is stopped automatically and explained.

In [ ]:
%pip install -q loop-autopsy  # skip this line if it is already installed

In [2]:
%load_ext loop_autopsy

loop-autopsy 0.3.0 is watching your loops. Type %autopsy help for options.


## 1. A stuck loop
The classic mistake: forgetting to update the counter.

In [3]:
count = 1
while count <= 5:
    print(count)

1
1


round,count (same)
1,1
2,1
3,1


## 2. A loop that goes the wrong way

In [4]:
i = 0
while i < 10:
    i -= 1

round,i
496,-495
497,-496
498,-497
499,-498
500,-499
501,-500


## 3. A loop that jumps over its stop value
Floats are never exactly `1.0` here (Lecture 2: floating point numbers).

In [5]:
x = 0.0
while x != 1.0:
    x += 0.1

round,x
107,10.599999999999978
108,10.699999999999978
109,10.799999999999978
110,10.899999999999977
111,10.999999999999977
112,11.099999999999977


## 4. A loop stuck in a cycle

In [6]:
a = 0
while a < 5:
    a = (a + 1) % 3

round,a
2,1
3,2
4,0
5,1
6,2
7,0


## 5. Looping over a list while adding to it

In [7]:
nums = [1, 2, 3]
for n in nums:
    nums.append(n)

round,nums,n
"1,928",list(len=1930),2
"1,944",list(len=1946),3
"1,960",list(len=1962),1
"1,976",list(len=1978),2
"1,992",list(len=1994),3
"2,008",list(len=2010),1


## 5b. Runaway recursion
A function that keeps calling itself is the recursive cousin of an infinite loop. Instead of a traceback hundreds of lines long, you get the arguments of the first and last calls.

In [8]:
def countdown(n):
    if n == 0:
        return 'done'
    return countdown(n - 2)

countdown(5)

call,n
1,5
2,3
3,1
975,-1943
976,-1945
977,-1947


## 5c. Mistakes read from the code
Some mistakes can be seen in the code itself. loop-autopsy points at the exact line.

The update line is there, but it is not indented, so it runs only after the loop:

In [9]:
count = 1
while count <= 5:
    print(count)
count += 1

1
1


round,count (same)
1,1
2,1
3,1


A bare `except:` catches **everything**, including loop-autopsy's stop signal and the stop button ■. Around a loop, it hides why the loop was stopped:

In [10]:
try:
    total = 0
    while total < 10:
        print('adding...')  # forgot total += 1
except:
    print('Something went wrong')

[loop-autopsy] Note: this loop runs inside a `try` whose `except:` (line 5) also catches loop-autopsy's stop signal and the stop button ■, so if the loop is stopped you will not see why. Write `except Exception:` instead: it still catches errors but lets a stop through. `%autopsy last` shows a report that was hidden this way.
adding...
adding...
Something went wrong


`%autopsy last` still shows the report that the `except:` hid:

In [11]:
%autopsy last

round,total (same)
1,0
2,0
3,0


`itertools.count()` counts forever. With no `break`, this loop can only end with an error, so you get a note on its first round. (The time limit is made short here so the demo does not wait a minute.)

In [12]:
%autopsy config time_limit=3
import itertools

for n in itertools.count(1):
    square = n * n

time_limit = 3.0
[loop-autopsy] Note: `for n in itertools.count(1):` (line 4) loops over `itertools.count`, which never runs out, and has no `break`, `return` or `raise` inside, so it can only end with an error. Add a `break` for when the work is done; until then loop-autopsy stops it after the 3 s time limit, or press ■.


round,square,n
"5,528,488",30564168509169,5528488
"5,528,744",30566999160049,5528744
"5,529,000",30569829942001,5529000
"5,529,256",30572660855025,5529256
"5,529,512",30575491899121,5529512
"5,529,768",30578323074289,5529768


Put the time limit back:

In [13]:
%autopsy reset

Settings are back to their defaults:
  time_limit       3.0 -> 60.0


## 6. Normal loops are not touched

In [14]:
n = 27
steps = 0
while n != 1:
    n = n // 2 if n % 2 == 0 else 3 * n + 1
    steps += 1
print('Collatz steps:', steps)

Collatz steps: 111


## 7. The cell health bar
`auto` mode only shows it for slow or memory-heavy cells. Turn it on to see it for every cell.

In [15]:
%autopsy heatmap on
big = [list(range(1000)) for _ in range(2000)]
len(big)

Cell health bar: on


2000

### Which variables use the memory?

`%autopsy memory` shows how much memory the kernel uses and its biggest variables, on demand. Add a number to see more: `%autopsy memory 10`.

In [16]:
%autopsy memory 3

Memory: 124.4 MB of 15.6 GB (1%)

variable     size  of limit  type
--------  -------  --------  ----------------
big       68.8 MB      0.4%  list (len 2,000)
nums        71 KB     <0.1%  list (len 2,010)
square       32 B     <0.1%  int

In [17]:
%autopsy heatmap auto
del big

Cell health bar: auto


### The whole session at a glance
`%autopsy summary` shows every cell you have run as a coloured square, then the heaviest cells first.
A cell is coloured by what it did itself: its run time and the memory it added.
Use `%autopsy summary all` for every cell in run order.

In [18]:
%autopsy summary

cell,health,time,memory,total after,loops,notes,first line
[12],watch out,3.05 s,+4 KB,63.6 MB,1,1 stopped,%autopsy config time_limit=3
[8],healthy,0.42 s,+1.1 MB,63.6 MB,0,error,def countdown(n):
[3],healthy,0.21 s,+3.0 MB,62.1 MB,1,1 stopped,count = 1
[15],healthy,0.12 s,+60.7 MB,124.4 MB,0,,%autopsy heatmap on
[17],healthy,0.05 s,-59.0 MB,65.4 MB,0,,%autopsy heatmap auto
[7],healthy,0.02 s,+108 KB,62.5 MB,1,1 stopped,"nums = [1, 2, 3]"
[4],healthy,0.01 s,+52 KB,62.2 MB,1,1 stopped,i = 0
[9],healthy,0.01 s,+24 KB,63.6 MB,1,1 stopped,count = 1
[5],healthy,0.01 s,+104 KB,62.3 MB,1,1 stopped,x = 0.0
[6],healthy,0.01 s,+92 KB,62.4 MB,1,1 stopped,a = 0


## 8. See the last report again, or skip watching for one cell

In [19]:
%autopsy last

round,square,n
"5,528,488",30564168509169,5528488
"5,528,744",30566999160049,5528744
"5,529,000",30569829942001,5529000
"5,529,256",30572660855025,5529256
"5,529,512",30575491899121,5529512
"5,529,768",30578323074289,5529768


In [20]:
%%autopsy off
q = 0
while q < 100_000:
    q += 1
print('ran without watching:', q)

ran without watching: 100000


In [21]:
%autopsy

loop-autopsy is ON. Settings:
  time_limit       60.0
  estimate_after   5.0
  direction_after  500
  jump_after       100
  growth_after     2000
  memory_stop      0.85
  memory_warn      0.75
  max_output_lines 1000
  max_displays     200
  muted_tail       20
  replay_rows      6
  max_watched      25
  heatmap          auto
  time_yellow      1.0
  time_red         10.0
  mem_yellow       0.5
  mem_red          0.8
  history_cells    200
  grow_yellow      0.05
  grow_red         0.2
  rerun_growth_mb  10
